In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
# Check for GPU
device = torch.device('cuda')
print(f"Using device: {device}")

Using device: cuda


In [4]:
df = pd.read_csv('../Data/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# train test split

X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        # Convert to PyTorch tensors
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [ ]:
# Objective function
def objective(trial):
    
    # Next hyperparameter values from the search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)


In [14]:
class MyNN(nn.Module):

    def __init__(self, num_features):

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)


In [15]:
learning_rate = 0.1
epochs = 100

In [16]:
# instatiate the model
model = MyNN(X_train.shape[1])
model = model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(), lr= learning_rate, weight_decay=1e-4)

In [17]:
print(f"Training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Total samples trained: {len(train_dataset) * epochs}")


Training samples: 48000
Batches per epoch: 1500
Total samples trained: 4800000


In [18]:
# training loop

for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:

        # Move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # forward pass
        outputs = model(batch_features)

        # calculate loss
        loss = criterion(outputs, batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 0.6255791780451934
Epoch: 2 , Loss: 0.49239900889992716
Epoch: 3 , Loss: 0.4554253513664007
Epoch: 4 , Loss: 0.4297527638177077
Epoch: 5 , Loss: 0.41631936643024287
Epoch: 6 , Loss: 0.403158124084274
Epoch: 7 , Loss: 0.3939924410879612
Epoch: 8 , Loss: 0.3832652302533388
Epoch: 9 , Loss: 0.37318766076366106
Epoch: 10 , Loss: 0.36861190950870515
Epoch: 11 , Loss: 0.36521197732786337
Epoch: 12 , Loss: 0.35459895703196526
Epoch: 13 , Loss: 0.3464978189269702
Epoch: 14 , Loss: 0.3447459938029448
Epoch: 15 , Loss: 0.3429492408335209
Epoch: 16 , Loss: 0.33688249460111064
Epoch: 17 , Loss: 0.33259921867152054
Epoch: 18 , Loss: 0.33094904050727686
Epoch: 19 , Loss: 0.32907037404924633
Epoch: 20 , Loss: 0.32713597065707045
Epoch: 21 , Loss: 0.3186058467378219
Epoch: 22 , Loss: 0.31940535570681095
Epoch: 23 , Loss: 0.3208069143096606
Epoch: 24 , Loss: 0.3153081162944436
Epoch: 25 , Loss: 0.3139980726291736
Epoch: 26 , Loss: 0.3127422924116254
Epoch: 27 , Loss: 0.3094462766150633

In [19]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [20]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        # move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()

print(f"Test Accuracy: {(correct / total) * 100:.2f}%")

Test Accuracy: 88.85%


In [21]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()

print(f"Train Accuracy: {(correct / total) * 100:.2f}%")

Train Accuracy: 94.06%
